# SoftConstruct Jobs Scraper

**Source:** [peopleforce.softconstruct.com/careers](https://peopleforce.softconstruct.com/careers)  
**Platform:** PeopleForce ATS (server-side rendered HTML — no JavaScript required)  
**Filter:** Yerevan / Armenia positions only

### Why SoftConstruct separately?
SoftConstruct is one of Armenia's largest IT/iGaming companies (~3,000+ employees). Their jobs are posted exclusively on their own PeopleForce-powered careers portal, not on LinkedIn or local job boards.

### Scraping strategy
1. Paginate through all listing pages (`?page=N`, 10 jobs/page, 20 pages total = 196 jobs)
2. Each card has: job title (`h4 > a.stretched-link`) + department & location (`div.tw-text-dark-neutral-80`)
3. Filter cards where location = "Yerevan" (visible in listing — no detail page needed for filtering)
4. Fetch each Armenia job's detail page for full description text

### Page structure
- Listing: `div.card-body` → title link + `"Department, Location"` metadata
- Detail: plain HTML, full content directly in `<body>` — no JS rendering needed

### Ethics & robots.txt
- PeopleForce careers portals are public by design (no login required)
- Rate-limited: 1.0 s between detail page requests, 0.5 s between listing pages

### Output files
- `data/raw/jobs/softconstruct_jobs_raw.csv` — 152 rows
- `data/processed/jobs/softconstruct_jobs_standardized.csv` — 152 rows

In [1]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

BASE    = "https://peopleforce.softconstruct.com"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                  "Chrome/122.0.0.0 Safari/537.36"
}
DELAY_LISTING = 0.5
DELAY_DETAIL  = 1.0

BASE_DIR = Path.cwd()
RAW_DIR  = BASE_DIR / "data" / "raw" / "jobs"
PROC_DIR = BASE_DIR / "data" / "processed" / "jobs"

print(f"Base URL: {BASE}/careers")
print(f"Raw output:  {RAW_DIR}")
print(f"Proc output: {PROC_DIR}")

Base URL: https://peopleforce.softconstruct.com/careers
Raw output:  /Users/lianaaghamalyan/thesis_data/data/raw/jobs
Proc output: /Users/lianaaghamalyan/thesis_data/data/processed/jobs


## Helper functions

In [2]:
def html_to_text(html_str):
    if not html_str: return ""
    text = BeautifulSoup(str(html_str), "html.parser").get_text("\n", strip=True)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

BOILERPLATE = {
    "SOFTCONSTRUCT", "Open Positions", "Home", "Apply now", "Share", "Link",
    "Share to", "Powered by", "PeopleForce", "English", "Українська", "Polski",
    "Español", "Português", "Deutsch", "Русский",
}

def clean_detail_text(soup):
    """Strip nav/footer/boilerplate from detail page and return clean text."""
    for tag in soup.find_all(["nav", "footer", "header"]):
        tag.decompose()
    for tag in soup.find_all(True, class_=lambda c: c and any(
            kw in " ".join(c).lower() for kw in ["nav", "footer", "header", "share", "powered"])):
        tag.decompose()
    body = soup.find("body")
    raw = body.get_text("\n", strip=True) if body else ""
    lines = [l for l in raw.split("\n")
             if l.strip() and l.strip() not in BOILERPLATE
             and not l.strip().startswith("SOFTCONSTRUCT -")]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()

print("Helpers defined.")

Helpers defined.


## Step 1 — Paginate listing pages and collect all job cards

In [3]:
all_cards = []
page = 1
while True:
    url  = f"{BASE}/careers?page={page}"
    soup = BeautifulSoup(requests.get(url, headers=HEADERS, timeout=20).text, "html.parser")
    cards = soup.find_all("div", class_="card-body")
    if not cards:
        break
    for card in cards:
        a    = card.find("a", class_="stretched-link")
        meta = card.find("div", class_="tw-text-dark-neutral-80")
        if not a: continue
        href      = a["href"]
        title     = a.get_text(strip=True)
        meta_text = meta.get_text(strip=True) if meta else ""
        dept = meta_text.rsplit(",", 1)[0].strip() if "," in meta_text else meta_text
        loc  = meta_text.rsplit(",", 1)[1].strip() if "," in meta_text else ""
        all_cards.append({
            "url": BASE + href if href.startswith("/") else href,
            "title": title, "department": dept, "location": loc,
        })
    print(f"  Page {page}: {len(cards)} cards | total: {len(all_cards)}")
    if len(cards) < 10: break
    page += 1
    time.sleep(DELAY_LISTING)

armenia = [c for c in all_cards
           if "yerevan" in c["location"].lower() or "armenia" in c["location"].lower()]
print(f"\nTotal cards: {len(all_cards)}")
print(f"Armenia/Yerevan jobs: {len(armenia)}")

  Page 1: 10 cards | total: 10
  Page 2: 10 cards | total: 20
  Page 3: 10 cards | total: 30
  Page 4: 10 cards | total: 40
  Page 5: 10 cards | total: 50
  Page 6: 10 cards | total: 60
  Page 7: 10 cards | total: 70
  Page 8: 10 cards | total: 80
  Page 9: 10 cards | total: 90
  Page 10: 10 cards | total: 100
  Page 11: 10 cards | total: 110
  Page 12: 10 cards | total: 120
  Page 13: 10 cards | total: 130
  Page 14: 10 cards | total: 140
  Page 15: 10 cards | total: 150
  Page 16: 10 cards | total: 160
  Page 17: 10 cards | total: 170
  Page 18: 10 cards | total: 180
  Page 19: 10 cards | total: 190
  Page 20: 6 cards | total: 196

Total cards: 196
Armenia/Yerevan jobs: 152


## Step 2 — Scrape detail pages for full job descriptions

In [4]:
records = []
for i, c in enumerate(armenia, 1):
    print(f"[{i}/{len(armenia)}] {c['title'][:60]}")
    r    = requests.get(c["url"], headers=HEADERS, timeout=20)
    soup = BeautifulSoup(r.text, "html.parser")
    full_text = clean_detail_text(soup)
    records.append({
        "source":       "softconstruct",
        "source_url":   c["url"],
        "job_title":    c["title"],
        "company_name": "SoftConstruct",
        "location":     c["location"],
        "department":   c["department"],
        "posting_date": "",
        "full_text":    full_text,
    })
    print(f"  full_text: {len(full_text)} chars")
    time.sleep(DELAY_DETAIL)

[1/152] Golang Developer
  full_text: 1209 chars
[2/152] Customer Support Specialist (Arabic and English Languages)
  full_text: 1513 chars
[3/152] Middle HR Administrative Specialist / VBET
  full_text: 2387 chars
...
[152/152] Customer Support Agent (Turkish)/VBET - Mixed shifts
  full_text: 2572 chars


## Step 3 — Save outputs

In [5]:
df = pd.DataFrame(records)
raw_path = RAW_DIR / "softconstruct_jobs_raw.csv"
df.to_csv(raw_path, index=False, encoding="utf-8")
print(f"Raw saved: {raw_path} ({len(df)} rows)")

std_cols = ["source","source_url","job_title","company_name","location","department","posting_date","full_text"]
std_df = df[std_cols]
std_path = PROC_DIR / "softconstruct_jobs_standardized.csv"
std_df.to_csv(std_path, index=False, encoding="utf-8")
print(f"Standardized saved: {std_path} ({len(std_df)} rows)")
print()
print("=== FIELD COVERAGE ===")
for col in std_cols:
    filled = std_df[col].replace("", pd.NA).notna().sum()
    print(f"  {col:20s}: {filled}/{len(std_df)} ({filled/len(std_df)*100:.0f}%)")
print()
ft = std_df["full_text"].str.len()
print(f"full_text — Min: {ft.min()}  Median: {ft.median():.0f}  Max: {ft.max()}")
print()
print("Note: posting_date not displayed on PeopleForce careers portal.")

Raw saved: /Users/lianaaghamalyan/thesis_data/data/raw/jobs/softconstruct_jobs_raw.csv (152 rows)
Standardized saved: /Users/lianaaghamalyan/thesis_data/data/processed/jobs/softconstruct_jobs_standardized.csv (152 rows)

=== FIELD COVERAGE ===
  source              : 152/152 (100%)
  source_url          : 152/152 (100%)
  job_title           : 152/152 (100%)
  company_name        : 152/152 (100%)
  location            : 152/152 (100%)
  department          : 152/152 (100%)
  posting_date        : 0/152 (0%)
  full_text           : 152/152 (100%)

full_text — Min: 141  Median: 2202  Max: 5565

Note: posting_date not displayed on PeopleForce careers portal.
